# This is a notebook for querying the SeaDataNet CDI Open API
-   **It is recommended to start using the new SeaDataNet CDI Beacon Subsetting API**
-   You can run each cell individually by pressing "shift + enter".
-   For more information, questions, bugs, please contact us [info@maris.nl](mailto:info@maris.nl)

#### In order to get access to the CDI API, you need to fill in your unique personal token between the "quotes" in the cell below.


In [ ]:
CdiOpenToken = "To request a token, contact paul at maris.nl -> maris.nl/contact"

#### Install the required packages, if you have not already installed them in your environment:

-   `pip install -r requirements_seadatanet.txt`

#### Import the required packages


In [114]:
import time
import requests
import json
import zipfile
import os
from SPARQLWrapper import SPARQLWrapper, JSON

### Let's configure our filter fields:

In [117]:
# EXV Input
exv_code = "EXV017"
mindate = "2010"  # yyyymmdd
maxdate = "201001"  # yyyymmdd
minlon = -180
maxlon = 180
minlat = -90
maxlat = 90

### This function converts our EXV to P01:

In [118]:
def EXV_iadopt(exv_code):
    # SPARQL endpoint
    endpoint_url = "https://vocab.nerc.ac.uk/sparql/"

    # Construct full identifier
    exv_identifier = f"SDN:EXV::{exv_code}"

    # Create the query with the user input
    query = f"""
    PREFIX dce: <http://purl.org/dc/elements/1.1/>
    PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
    PREFIX iadopt: <https://w3id.org/iadopt/ont#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

    SELECT DISTINCT ?p02 ?prefLabel ?notation
    WHERE {{
      ?exv a skos:Concept .
      ?exv dce:identifier "{exv_identifier}" .

      OPTIONAL {{?exv iadopt:hasApplicableMatrix ?matrix .}}
      ?exv iadopt:hasApplicableObjectOfInterest ?ooi .
      ?exv iadopt:hasApplicableProperty ?property .

      <http://vocab.nerc.ac.uk/collection/P01/current/> skos:member ?p01 .

      OPTIONAL {{ ?p01 iadopt:hasMatrix ?matrix . }}
      ?p01 iadopt:hasObjectOfInterest ?ooi .
      ?p01 iadopt:hasProperty ?property .

        <http://vocab.nerc.ac.uk/collection/P02/current/> skos:member ?p02 .
      ?p01 skos:broader ?p02 .

      OPTIONAL {{ ?p02 skos:prefLabel ?prefLabel . }}
      OPTIONAL {{ ?p02 skos:notation ?notation . }}
    }}
    """

    # Set up the SPARQL request
    sparql = SPARQLWrapper(endpoint_url)
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)

    # Run the query and parse results
    results = sparql.query().convert()

    codes = []

    # Show results
    for result in results["results"]["bindings"]:
        uri = result.get("p02", {}).get("value", "")
        codes.append(uri.rstrip("/").split("/")[-1])

    matching_p02s = []

    for code in codes:
        if code in params:
            matching_p02s.append(code)

    return matching_p02s

matching_p02s = EXV_iadopt(exv_code)
print(matching_p02s)

['TEMP']


#### This funciton will create the query body based on your input parameters.

-  To find more query fields and API capabilities, please find the OpenAPI description here: https://cdi-open.seadatanet.org/api


In [119]:
def create_query(mindate, maxdate, minlon, maxlon, minlat, maxlat, matching_p02s: list[str]):
    body = {
        'user_order_name': 'My CDI API Notebook order',
        'motivation': 'not applicable (only open data)',
        'offset': 0, 
        'data_format_L24': 'cfpoint',
        'query_fields': {
            'start_date': mindate,
            'end_date': maxdate,
            'north': maxlat,
            'east': maxlon,
            'south': minlat,
            'west': minlon,
            'parameters_p02': ','.join(matching_p02s).lower()
        }
    }
    return body

query_body = create_query(mindate, maxdate, minlon, maxlon, minlat, maxlat, matching_p02s)

print("QUERY:", json.dumps(query_body))

QUERY: {"user_order_name": "My CDI API Notebook order", "motivation": "not applicable (only open data)", "offset": 0, "data_format_L24": "cfpoint", "query_fields": {"start_date": "2010", "end_date": "201001", "north": 90, "east": 180, "south": -90, "west": -180, "parameters_p02": "temp"}}


#### This function creates an order in CDI Open, when the search query returns more than 10k datasets, we create an order for each 10k datasets.


In [120]:
def create_order(_query_body):
    """
    Create an order in CDI Open API. 
    The order is created with the query body passed as an argument.
    Returns:
        (order_number, has_more_data, number_of_hits): A tuple containing the order number, a boolean indicating if there is more data to order, and the number of hits from the query.
    """
    response = requests.post("https://cdi-open.seadatanet.org/api/order/query", json.dumps(_query_body),
        headers={"Authorization": f"Bearer {CdiOpenToken}",
                "Content-type": "application/json"})

    if response.status_code == 401:
        print("Unauthorized: Your token is invalid or expired, please use a token connected to a CDI Open API account.")
        return (None, False, 0)
    
    data = json.loads(response.text)

    if 'OrderNumber' in data:
        order_number = data['OrderNumber']
        order_number_lines = data['OrderNumberLines']
        number_of_hits_from_query = data['NumberOfHitsFromQuery']
        return (order_number, order_number_lines < number_of_hits_from_query, number_of_hits_from_query)

    # check if data.warnings exists:
    if 'warnings' in data:
        for warning in data['warnings']:
           print(warning['subject'])

    return (None, False, 0)
            

moreDataToOrder = True
orderNumbers = []

while moreDataToOrder:
    order_number, moreDataToOrder, hits = create_order(query_body)

    if order_number is not None:
        print(f"Order created: {order_number}, hits: {hits}, more data to order: {moreDataToOrder}")
        orderNumbers.append(order_number)
    else:
        print("No order number returned.")
        break

    # Check if there is more data to order
    if not moreDataToOrder:
        print("No more data to order.")
        break

    query_body['offset'] = query_body['offset'] + 10000

print(orderNumbers)
    


Order created: 73238, hits: 4361, more data to order: False
No more data to order.
[73238]


### Because this API creates orders/jobs, we have to poll if the order is finished:

In [122]:
def get_order_status(order_number):
    url = f"https://cdi-open.seadatanet.org/api/order/{order_number}"

    response = requests.get(url, headers={"Authorization": f"Bearer {CdiOpenToken}",
                "Content-type": "application/json"})
    
    data = json.loads(response.text)

    if 'orderLines' in data:
        order_lines = data['orderLines']
        order_lines_processing = data['orderLinesProcessing']
        order_lines_ready_for_download = data['orderLinesReadyForDownload']

        percentage_ready = (order_lines_ready_for_download / order_lines) * 100

        if percentage_ready >= 99 and order_lines_ready_for_download > 0:
            return True
    
    return False

doneOrders = []

while True:
    allDone = True

    for orderNumber in orderNumbers:
        done = get_order_status(orderNumber)
        if done:
            if orderNumber not in doneOrders:
                print(f"Order {orderNumber} is done.")
                doneOrders.append(orderNumber)
        else:
            allDone = False


    if allDone:
        break

    time.sleep(1)
    



Order 73238 is done.


### When the order is done, we download the .zip files (data & metadata) to disk

In [ ]:
def download_to_disk(url, filename, stream=True, headers=[]):
    print(f"Downloading {url} to {filename}... ")

    try:

        if stream:
            # Perform the GET request with streaming
            with requests.get(url, stream=True, headers=headers) as response:
                response.raise_for_status()  # Raise an exception for HTTP errors
                with open(filename, 'wb') as file:
                    for chunk in response.iter_content(chunk_size=8192):  # Download in 8 KB chunks
                        if chunk:  # Filter out keep-alive chunks
                            file.write(chunk)

        else:
            # Perform the GET request without streaming
            response = requests.get(url, headers=headers)
            response.raise_for_status()
            with open(filename, 'w') as file:
                file.write(response.text)

        print(f"Downloaded {filename} successfully.")
        return True
    
    except Exception as e:
        print(f"Error: {e}")

    return False


def download_order(order_number):
    url = f"https://cdi-open.seadatanet.org/api/order/{order_number}"

    response = requests.get(url, headers={"Authorization": f"Bearer {CdiOpenToken}",
                "Content-type": "application/json"})
    
    data = json.loads(response.text)
    

    if 'downloads' in data:
        download_links = []
        
        if not os.path.exists('data'):
            os.makedirs('data')

        for download in data['downloads']:
            download_filename = 'data/' + download['unrestricted']['data']['name']
            download_link = download['unrestricted']['data']['downloadUrl']
            csv_filename = download_filename.replace('.zip', '_csv.zip')
            csv_link = download['unrestricted']['csv']['downloadUrl']

            download_links.append((download_link, download_filename, csv_link, csv_filename))

            download_to_disk(download_link, download_filename, headers={"Authorization": f"Bearer {CdiOpenToken}"})
            download_to_disk(csv_link, csv_filename, headers={"Authorization": f"Bearer {CdiOpenToken}"})

        return download_links


    print(f"No downloads available for order {order_number}.")

    return None

results = dict()

for orderNumber in doneOrders:
    results[orderNumber] = download_order(orderNumber)

print("Downloaded files:")

for orderNumber in results:
    print(f"Order {orderNumber}:")
    for download in results[orderNumber]:
        download_link, download_filename, csv_link, csv_filename = download

        print(f"  {download_filename} ({download_link})")
        print(f"  {csv_filename} ({csv_link})")





Downloaded data/order_73230_unrestricted.zip successfully.
Downloaded data/order_73230_unrestricted_csv.zip successfully.
Downloaded files:
Order 73230:
  data/order_73230_unrestricted.zip (https://seadata.cineca.it/api/orders/73230/download/00/c/gAAAAABn9m4lZp6w_3fOw4mlR8fMTNJbwn2VXt4ZGYq1jM8yWRBSZnJI1LWFZoqRokga_Rt73pCcmMwpmf_qRSJGM4DDxWOQPSSzF78PBkqpBj16C8ni4tEHTK-zf5YNWMfAUm3mwd0P)
  data/order_73230_unrestricted_csv.zip (https://cdi-open.seadatanet.org/api/order/download/csv/73230/unrestricted)


### After downloading, unzip the files to their respective directory.

In [ ]:
# after downloading the zips, we want to unzip them all:
def unzip_file(zip_filepath, extract_to):
    """
    Unzips a file to the specified directory.
    
    Args:
        zip_filepath (str): Path to the .zip file.
        extract_to (str): Directory where the contents will be extracted.
    """
    try:
        with zipfile.ZipFile(zip_filepath, 'r') as zip_ref:
            zip_ref.extractall(extract_to)
            print(f"Extracted {zip_filepath} to {extract_to}")
    except Exception as e:
        print(f"Error unzipping file {zip_filepath}: {e}")

for orderNumber in results:
    for download in results[orderNumber]:
        download_link, download_filename, csv_link, csv_filename = download

        target_directory = download_filename.replace('.zip', '')

        if not os.path.exists(target_directory):
            os.mkdir(target_directory)

        unzip_file(download_filename, target_directory)
        os.remove(download_filename)

        unzip_file(csv_filename, target_directory)
        os.remove(csv_filename)



Extracted data/order_73230_unrestricted.zip to data/order_73230_unrestricted
Extracted data/order_73230_unrestricted_csv.zip to data/order_73230_unrestricted


### This is where this notebook stops, the datasets have been downloaded now. 

This was a demonstration of the CDI API, it's recommended to start using the Beacon API in the future, as it's more powerful in it's capabilities (subsetting, filtering, data transformation & homogenization)